## ENTREGABLE 2 - HERRERA TORRES, DANIEL

El código procesa documentos PDF y utiliza la API de OpenAI para analizar su contenido, extrayendo información relevante como el título, autores, año de publicación y resumen. Finalmente, organiza estos datos en un DataFrame que se puede mostrar o manipular en Python.

In [4]:
# Importar la biblioteca load_dotenv para cargar variables de entorno desde un archivo .env
from dotenv import load_dotenv
import os # Importar os para trabajar con variables de entorno y rutas del sistema

# Especificar la ruta al archivo .env que contiene la API key
dotenv_path = "/Users/danielherrera/Documents/UNAV/TRD/.env"

# Cargar las variables de entorno desde el archivo .env en la ruta especificada
load_dotenv(dotenv_path)

# Obtener la API key almacenada en la variable de entorno OPENAI_API_KEY
api_key = os.getenv("OPENAI_API_KEY")
# Verificar si la API key se cargó correctamente desde el archivo .env
if api_key:
    print("API key cargada correctamente.") # Si la API key está disponible, imprimir un mensaje indicando que se cargó correctamente
else:
    print(f"API key no encontrada en {dotenv_path}") # Si la API key no se encuentra, mostrar un mensaje indicando que no se pudo cargar

API key cargada correctamente.


In [2]:
# Importar bibliotecas necesarias
import os
import openai
import pandas as pd
from dotenv import load_dotenv
from PyPDF2 import PdfReader

# -----------------------
# PASO 1: CARGAR API KEY
# -----------------------

# Cargar la API key desde el archivo .env
load_dotenv() 
openai.api_key = os.getenv("OPENAI_API_KEY") # Obtiene la clave de API desde las variables de entorno

# Validar si la API key fue cargada correctamente
if not openai.api_key:
    raise ValueError("API key no encontrada. Asegúrate de que esté configurada correctamente en el archivo .env.")

# -----------------------
# PASO 2: DEFINIR LA RUTA DE LOS ARCHIVOS PDF
# -----------------------

# Ruta de la carpeta con los PDFs de los que extrae información el código
pdf_folder_path = "/Users/danielherrera/Documents/UNAV/TRD/PDFs"

# -----------------------
# PASO 3: FUNCIONES PARA PROCESAR LOS PDFs
# -----------------------

# Función para extraer texto completo de un archivo PDF
def extract_text_from_pdf(pdf_path):
    """
    Lee un archivo PDF y extrae todo su contenido como texto.

    Args:
        pdf_path (str): Ruta completa al archivo PDF.

    Returns:
        str: Texto completo extraído del PDF.
    """
    text = "" # Inicializar una variable para el texto extraído
    try:
        reader = PdfReader(pdf_path) # Crear un lector de PDF
        for page in reader.pages: # Iterar sobre cada página del PDF
            text += page.extract_text() + "\n" # Extraer y concatenar el texto de cada página
    except Exception as e:
        print(f"Error procesando {pdf_path}: {e}") # Manejar errores de procesamiento
    return text

# Función para dividir texto en fragmentos manejables para la API
def split_text(text, max_tokens=3000):
    """
    Divide un texto en fragmentos de longitud manejable.

    Args:
        text (str): Texto completo a dividir.
        max_tokens (int): Máximo número de caracteres por fragmento.

    Returns:
        list: Lista de fragmentos de texto.
    """
    words = text.split()  # Dividir el texto en palabras
    chunks = []  # Lista para almacenar fragmentos
    current_chunk = []  # Fragmento actual
    current_length = 0  # Longitud actual del fragmento

    for word in words:  # Iterar sobre cada palabra
        current_length += len(word) + 1  # Considerar la longitud de la palabra más un espacio
        if current_length > max_tokens:  # Si el fragmento supera el límite
            chunks.append(" ".join(current_chunk))  # Agregar el fragmento a la lista
            current_chunk = []  # Reiniciar el fragmento actual
            current_length = len(word) + 1  # Iniciar longitud con la palabra actual
        current_chunk.append(word)  # Agregar palabra al fragmento actual

    if current_chunk: # Agregar el último fragmento si queda contenido
        chunks.append(" ".join(current_chunk))

    return chunks

# -----------------------
# PASO 4: FUNCIONES PARA CONSULTAR LA API DE OPENAI
# -----------------------

# Función para realizar las consultas a la API de OpenAI
def ask_openai(question, context):
    """
    Realiza una consulta a la API de OpenAI basada en un contexto y una pregunta.

    Args:
        question (str): Pregunta que se realizará a la API.
        context (str): Fragmento de texto que sirve como contexto.

    Returns:
        str: Respuesta proporcionada por la API.
    """
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4", # Modelo de OpenAI a utilizar
            messages=[
                {"role": "system", "content": "Eres un experto en análisis de documentos científicos."}, # Contexto general
                {"role": "user", "content": f"Contexto: {context}\nPregunta: {question}"} # Pregunta con contexto 
            ],
            max_tokens=200, # Máximo número de tokens en la respuesta
            temperature=0 # Respuestas determinísticas
        )
        return response["choices"][0]["message"]["content"].strip()  # Retornar respuesta 
    except Exception as e:
        print(f"Error en la API de OpenAI: {e}") # Manejar errores de la API
        return "Error"
        
# -----------------------
# PASO 5: PROCESAR UN PDF Y EXTRAER INFORMACIÓN CLAVE
# -----------------------

# Función para extraer datos específicos de un PDF
def process_pdf(pdf_path):
        """
    Extrae datos clave (título, autores, año, resumen) de un archivo PDF.

    Args:
        pdf_path (str): Ruta completa al archivo PDF.

    Returns:
        dict: Diccionario con los datos extraídos.
    """
    text = extract_text_from_pdf(pdf_path)  # Extraer texto completo del PDF
    if not text.strip():  # Verificar si el texto está vacío
        return {"Documento": os.path.basename(pdf_path), "Error": "Texto no encontrado"}

    chunks = split_text(text) # Dividir el texto en fragmentos para que la API sea capaz de procesar la información contenida en cada PDF

    # Inicializar variables para los datos extraídos
    title, authors, publication_year, abstract = "", "", "", ""

    for chunk in chunks: # Iterar sobre cada fragmento
        if not title:
            title = ask_openai("¿Cuál es el título del documento?", chunk) # Consulta para extraer el título
        if not authors:
            authors = ask_openai("¿Quiénes son los autores del documento?", chunk) # Consulta para extraer los autores
        if not publication_year:
            publication_year = ask_openai("¿En qué año se publicó el documento?", chunk) # Consulta para extrar año de publicación
        if not abstract:
            abstract = ask_openai("¿Cuál es el resumen (abstract) del documento?", chunk) 3 #Consulta para extraer un resumen del doocumento PDF

        # Si ya obtuvimos todos los datos, salir del bucle
        if title and authors and publication_year and abstract:
            break
    # Retornar datos extraídos en un diccionario
    return {
        "Documento": os.path.basename(pdf_path),  # Nombre del archivo
        "Título": title,  # Título extraído
        "Autor": authors,  # Autores extraídos
        "Año de Publicación": publication_year,  # Año de publicación
        "Resumen": abstract  # Resumen extraído
    }

# -----------------------
# PASO 6: OBTENER LA LISTA DE ARCHIVOS PDF
# -----------------------

# Crear una lista con las rutas de todos los archivos PDF en la carpeta
pdf_files = [os.path.join(pdf_folder_path, f) for f in os.listdir(pdf_folder_path) if f.endswith(".pdf")]

# -----------------------
# PASO 7: PROCESAR LOS ARCHIVOS PDF Y CREAR EL DATAFRAME
# -----------------------

# Procesar cada archivo PDF y extraer la información clave para construir el DataFrame
data_list = [process_pdf(pdf) for pdf in pdf_files]
# Crear un DataFrame con los datos extraídos
df = pd.DataFrame(data_list)

# -----------------------
# PASO 8: MOSTRAR LOS RESULTADOS
# -----------------------

# Mostrar el DataFrame imprimiendolo en la consola
print("Resultados del análisis:")
print(df)

# Renderizar el DataFrame en Jupyter Notebook
display(df)

Multiple definitions in dictionary at byte 0x2aefa for key /MediaBox
Multiple definitions in dictionary at byte 0x2b23c for key /MediaBox
Multiple definitions in dictionary at byte 0x2b3dd for key /MediaBox
Multiple definitions in dictionary at byte 0x2b59b for key /MediaBox
Multiple definitions in dictionary at byte 0x2b89c for key /MediaBox
Multiple definitions in dictionary at byte 0x2bb75 for key /MediaBox
Multiple definitions in dictionary at byte 0x2be3e for key /MediaBox
Multiple definitions in dictionary at byte 0x2c117 for key /MediaBox
Multiple definitions in dictionary at byte 0x2c300 for key /MediaBox
Multiple definitions in dictionary at byte 0x2c4f6 for key /MediaBox
Multiple definitions in dictionary at byte 0x2c6b7 for key /MediaBox
Multiple definitions in dictionary at byte 0x2c8f8 for key /MediaBox
Multiple definitions in dictionary at byte 0x2cb79 for key /MediaBox
Multiple definitions in dictionary at byte 0x2cdfa for key /MediaBox
Multiple definitions in dictionary

Resultados del análisis:
   Documento                                             Título  \
0  file1.pdf  El título del documento es "Placebo stimulates...   
1  file2.pdf  El título del documento es "Application of Bay...   
2  file3.pdf  El título del documento es "Placebo Effects on...   
3  file4.pdf  El título del documento es "Putting the ‘Art’ ...   
4  file5.pdf  El título del documento es "Placebo and Nocebo...   

                                               Autor  \
0  Los autores del documento son Jeremy Seymour y...   
1  Los autores del documento son Marco Tommasi, G...   
2  Los autores del documento son Sara Magelssen V...   
3  Los autores del documento son Michael H. Berns...   
4  Los autores del documento son Nicole Corsi y L...   

                                  Año de Publicación  \
0  El documento no proporciona información sobre ...   
1    El documento se publicó el 23 de julio de 2018.   
2            El documento se publicó en el año 2021.   
3    El doc

,Documento,Título,Autor,Año de Publicación,Resumen
0,file1.pdf,"El título del documento es ""Placebo stimulates...",Los autores del documento son Jeremy Seymour y...,El documento no proporciona información sobre ...,El documento desarrolla la Teoría del Placebo ...
1,file2.pdf,"El título del documento es ""Application of Bay...","Los autores del documento son Marco Tommasi, G...",El documento se publicó el 23 de julio de 2018.,El documento no proporciona un resumen o abstr...
2,file3.pdf,"El título del documento es ""Placebo Effects on...",Los autores del documento son Sara Magelssen V...,El documento se publicó en el año 2021.,El documento no proporciona un resumen explíci...
3,file4.pdf,"El título del documento es ""Putting the ‘Art’ ...",Los autores del documento son Michael H. Berns...,El documento se publicó el 22 de julio de 2020.,El documento no proporciona un resumen o abstr...
4,file5.pdf,"El título del documento es ""Placebo and Nocebo...",Los autores del documento son Nicole Corsi y L...,El documento se publicó el 6 de marzo de 2017.,El documento no proporciona un resumen o abstr...


#### Problemas encontrados desarrollando el trabajo:

1- Se logro ejecutar el el codigo usando una LLM de HuggingFace, sin embargo el resultado del DataFrame nunca fue acorde a lo esperado. Por este problema se toma la decision de conectarse a la API de OpenAI que interactua con un LLM. Al conectarse a la API de OpenAI, a pesar de no ser estrictamente necesario, es recomendable esconder la llave de acceso a la API. Es por eso que se tomo la decision de cargar la API key desde un archivo .env utilizando la biblioteca dotenv.

2- La API de OpenAI rechazaba solicitudes porque los textos eran demasiado largos, excediendo los límites de tokens. Se desarrolló una función para dividir el texto en fragmentos (split_text) que se envían por separado a la API y se establecio un balance entre el tamaño de los fragmentos y las llamadas a la API.

3- El resultado del DataFrame a pesar de no ser perfecto, es bastante coherente respecto a la informacion de los PDF. Sin embargo el problema que sigue siendo persistente en la respuesta de la API es el preambulo antes de dar el dato en concreto, por ejemplo dice "El título del documento es: ..." o "El documento se publico en:..." Se intento resolver definiendo una funcion para una respuesta limpia, sin embargo no se logro conseguir el resultado esperado. La función que se intento usar fue la siguiente, sin embargo al no funcionar se elimino del código: 

##### Función para limpiar frases predefinidas
def clean_response(response):
    # Lista de frases a eliminar
    phrases_to_remove = [
        "El título del documento es:",
        "Los autores del documento son:",
        "El documento fue publicado en:",
        "El resumen del documento es:",
        "Título:",
        "Autores:",
        "Fecha de publicación:",
        "Resumen:"
    ]
    for phrase in phrases_to_remove:
        response = response.replace(phrase, "").strip()
    return response

#### Conclusiones
- Interacción eficiente con modelos de lenguaje requiere diseño cuidadoso y optimizado. El uso de modelos de lenguaje como GPT-4 a través de la API de OpenAI es una herramienta para extraer información de documentos científicos. Sin embargo, su implementación exitosa depende de dividir adecuadamente los textos largos en fragmentos manejables, para obtener datos claros y optimizar las preguntas enviadas a la API. Este proyecto demuestra la importancia de adaptar las estrategias a las limitaciones de tokens y tiempo de respuesta del modelo.

- El manejo de datos no estructurados requiere herramientas complementarias y manejo de errores robustos. La extracción de texto de documentos PDF complejos es un desafío debido a su diversidad estructural y calidad variable. Herramientas como PyPDF2 funcionan bien para documentos bien formateados, pero podrían ser insuficientes para casos más complicados. Por esto, agregar bloques de manejo de errores y considerar herramientas adicionales, como pdfplumber, puede mejorar la capacidad de procesar documentos de manera consistente.